# 08 — Heatmap Visualization (Paris)

Visualizes ML predictions as a heatmap overlaid on the Paris grid.

**Colors:**
- 🔵 Blue — Residential
- 🔴 Red — Commercial
- 🟡 Yellow — Industrial

**Output:**
- `outputs/Paris/08_heatmap_predictions.png` — static map
- `outputs/Paris/08_heatmap_interactive.html` — interactive Folium map

In [ ]:
PARIS_CONFIG = "paris.json"
PLOTS_DIR    = "outputs/Paris"

In [ ]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
import os
import math

os.makedirs(PLOTS_DIR, exist_ok=True)

with open(PARIS_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

CELL_SIZE_M = config["grid_cell_size_m"]
CSV_DIR     = config["csv_dir"]

df = pd.read_csv(f"{CSV_DIR}/07_predictions.csv", dtype={"cell_id": str})
print(f"Loaded {len(df)} cell predictions")
print(df["predicted_zone"].value_counts().to_string())

In [ ]:
# ── Cell dimensions ───────────────────────────────────
REF_LAT  = df["cell_lat"].mean()
LAT_STEP = CELL_SIZE_M / 111_000
LON_STEP = CELL_SIZE_M / (111_000 * math.cos(math.radians(REF_LAT)))
HALF_LAT = LAT_STEP / 2
HALF_LON = LON_STEP / 2

CLASS_COLORS = {
    "residential": "#2166AC",  # blue
    "commercial":  "#B2182B",  # red
    "industrial":  "#F4A736",  # amber/yellow
}

In [ ]:
# ── Static matplotlib heatmap ─────────────────────────
import contextily as ctx

fig, ax = plt.subplots(figsize=(12, 12))

for _, row in df.iterrows():
    base_color = CLASS_COLORS.get(row["predicted_zone"], "#999999")
    conf  = row.get("confidence", 0.7)
    alpha = 0.4 + 0.5 * conf
    rect  = mpatches.Rectangle(
        (row["cell_lon"] - HALF_LON, row["cell_lat"] - HALF_LAT),
        LON_STEP, LAT_STEP,
        linewidth=0.1, edgecolor="gray",
        facecolor=base_color, alpha=alpha
    )
    ax.add_patch(rect)

pad = 0.005
ax.set_xlim(df["cell_lon"].min() - pad, df["cell_lon"].max() + pad)
ax.set_ylim(df["cell_lat"].min() - pad, df["cell_lat"].max() + pad)
ax.set_aspect("equal")

try:
    ctx.add_basemap(ax, crs="EPSG:4326",
                    source=ctx.providers.CartoDB.PositronNoLabels,
                    zoom=13, alpha=0.4)
except Exception as e:
    print(f"Basemap failed: {e}")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

counts = df["predicted_zone"].value_counts()
pct_parts = " · ".join(
    f"{cls.capitalize()} {100 * counts.get(cls, 0) / len(df):.1f}%"
    for cls in ["residential", "commercial", "industrial"]
    if counts.get(cls, 0) > 0
)
ax.set_title(
    f"Paris Grid — Zone Predictions\n"
    f"{len(df)} cells ({CELL_SIZE_M}m) · {pct_parts}",
    fontsize=13
)

legend_patches = [
    mpatches.Patch(color=c, label=l.capitalize())
    for l, c in CLASS_COLORS.items() if l in counts.index
]
ax.legend(handles=legend_patches, loc="lower right", fontsize=11,
          framealpha=0.9, edgecolor="gray")

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/08_heatmap_predictions.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {PLOTS_DIR}/08_heatmap_predictions.png")

In [ ]:
# ── Interactive folium map ────────────────────────────
try:
    import folium
    from folium import Rectangle

    m = folium.Map(
        location=[REF_LAT, df["cell_lon"].mean()],
        zoom_start=13,
        tiles="CartoDB positron",
    )

    for _, row in df.iterrows():
        bounds = [
            [row["cell_lat"] - HALF_LAT, row["cell_lon"] - HALF_LON],
            [row["cell_lat"] + HALF_LAT, row["cell_lon"] + HALF_LON],
        ]
        color   = CLASS_COLORS.get(row["predicted_zone"], "#999999")
        conf    = row.get("confidence", 0.7)
        opacity = 0.4 + 0.5 * conf

        popup_text = (
            f"<b>{row['cell_id']}</b><br>"
            f"Actual: {row['zone_type']}<br>"
            f"Predicted: <b>{row['predicted_zone']}</b><br>"
            f"Confidence: {conf:.0%}"
        )

        Rectangle(
            bounds=bounds,
            color="gray", weight=0.3,
            fill=True, fill_color=color, fill_opacity=opacity,
            popup=folium.Popup(popup_text, max_width=200),
        ).add_to(m)

    legend_html = """
    <div style="position:fixed; bottom:30px; left:10px; z-index:1000;
                background:white; padding:10px; border-radius:5px;
                border:1px solid gray; font-size:13px;">
    <b>Zone Type</b><br>
    <span style="color:#B2182B;">&#9632;</span> Commercial<br>
    <span style="color:#2166AC;">&#9632;</span> Residential<br>
    <span style="color:#F4A736;">&#9632;</span> Industrial
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    html_path = f"{PLOTS_DIR}/08_heatmap_interactive.html"
    m.save(html_path)
    print(f"Saved: {html_path}")

except ImportError:
    print("folium not installed — skipping interactive map.")